# Point clouds

These are the array moves behind lidar preprocessing: bounds, nearest neighbors, and voxel centroids. Shapes stay `(N, 3)` unless a problem says otherwise.

For a large cloud, say what you would do in production (chunk the broadcast, or `scipy.spatial.cKDTree`) after the vectorized version works.


These cells are PyTorch. Geometry wants float64, so the setup cell calls `torch.set_default_dtype(torch.float64)`. The library default is float32, and a plane SVD or a Kabsch in float32 will miss a hand check that passes in float64.


In [ ]:
import torch

torch.set_default_dtype(torch.float64)
torch.set_printoptions(precision=4, sci_mode=False)


## 1. Bounds and unit-sphere normalization

Return the axis-aligned minimum, the axis-aligned maximum, and a normalized copy of the cloud.

Normalization subtracts the centroid and divides by the largest distance from that centroid, so the result sits in the unit ball and touches the sphere. If every point is the same, return the centered zeros and do not divide.

Hand check: `(0,0,0)`, `(2,0,0)`, `(0,2,0)` has min `(0,0,0)`, max `(2,2,0)`, a centered result, and max norm 1.


In [ ]:
import torch

def bounds_and_unit_sphere(points):
    """
    AABB min/max and a copy centered at the origin, scaled into the unit ball.

    The scale is the max distance from the centroid, so the result touches the
    unit sphere. A degenerate cloud (all points equal) is returned centered.
    """
    raise NotImplementedError


### Solution

Reference answer and checks. It replaces whatever you defined above.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_default_dtype(torch.float64)
DTYPE = torch.float64

def _t(x, dtype=DTYPE):
    if isinstance(x, torch.Tensor):
        return x.to(dtype=dtype)
    return torch.as_tensor(x, dtype=dtype)

def bounds_and_unit_sphere(points):
    """AABB min/max and a copy centered at the origin, scaled into the unit ball."""
    points = _t(points)
    mn = points.amin(dim=0)
    mx = points.amax(dim=0)
    centered = points - points.mean(dim=0)
    scale = torch.linalg.norm(centered, dim=1).amax()
    normalized = centered if scale < 1e-12 else centered / scale
    return mn, mx, normalized

def check_bounds_and_unit_sphere():
    pts = torch.tensor([[0.0, 0.0, 0.0], [2.0, 0.0, 0.0], [0.0, 2.0, 0.0]])
    mn, mx, normed = bounds_and_unit_sphere(pts)
    assert torch.allclose(mn, torch.zeros(3))
    assert torch.allclose(mx, torch.tensor([2.0, 2.0, 0.0]))
    assert torch.allclose(normed.mean(dim=0), torch.zeros(3), atol=1e-8)
    assert torch.allclose(torch.linalg.norm(normed, dim=1).amax(), torch.tensor(1.0))
    _, _, flat = bounds_and_unit_sphere(torch.ones(4, 3))
    assert torch.allclose(flat, torch.zeros(4, 3), atol=1e-8)
    print("passed")

check_bounds_and_unit_sphere()


## 2. Nearest neighbors

For every point in `A`, find the closest point in `B`. Return indices into `B` and Euclidean distances.

Do it with broadcasting, not a Python loop over points. Then say the memory cost out loud.

Hand check: `A = [(0,0,0), (1,0,0)]` and `B = [(0.1,0,0), (5,0,0), (0.9,0.1,0)]` matches indices `[0, 2]`.

<details><summary>Hint</summary>

`torch.cdist(A, B)` is the direct call. A broadcast difference has shape `(N, M, D)` if you build it yourself. `argmin` over dim 1. `N = M = 10000` is about 800 MB of float64 distances. Chunk `A`, or use a KD-tree, past that.

</details>


In [ ]:
import torch

def nearest_neighbors(A, B):
    """
    Brute-force nearest neighbor in B for every point of A.

    A: (N, D), B: (M, D)
    returns idx (N,) into B, and Euclidean distances (N,).
    Memory is O(N*M). Chunk A, or use a KD-tree, when that product is large.
    """
    raise NotImplementedError


### Solution

Reference answer and checks. It replaces whatever you defined above.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_default_dtype(torch.float64)
DTYPE = torch.float64

def _t(x, dtype=DTYPE):
    if isinstance(x, torch.Tensor):
        return x.to(dtype=dtype)
    return torch.as_tensor(x, dtype=dtype)

def nearest_neighbors(A, B):
    """Nearest neighbor in B for every point of A.

    A: (N, D), B: (M, D)
    returns idx (N,) into B, and Euclidean distances (N,).
    torch.cdist is the usual PyTorch spelling of the broadcast distance.
    Memory is still O(N*M).
    """
    A = _t(A)
    B = _t(B)
    if B.shape[0] == 0:
        raise ValueError("B is empty")
    dist = torch.cdist(A, B)
    idx = dist.argmin(dim=1)
    return idx, dist[torch.arange(A.shape[0]), idx]

def check_nearest_neighbors():
    A = torch.tensor([[0.0, 0.0, 0.0], [1.0, 0.0, 0.0]])
    B = torch.tensor([[0.1, 0.0, 0.0], [5.0, 0.0, 0.0], [0.9, 0.1, 0.0]])
    idx, dist = nearest_neighbors(A, B)
    assert torch.equal(idx, torch.tensor([0, 2]))
    assert torch.allclose(dist, torch.tensor([0.1, torch.sqrt(torch.tensor(0.02))]))
    print("passed")

check_nearest_neighbors()


## 3. Chamfer distance

Symmetric Chamfer distance, defined here as

`mean over A of (distance to nearest B) + mean over B of (distance to nearest A)`.

Some papers divide by two, and some sum squared distances. Say which one you implemented.

Hand check: `A = [(0,0,0), (10,0,0)]` and `B = [(0,0,0)]` gives `5`, because one side contributes 0 and the other contributes the average of 0 and 10. Two unit squares sitting one unit apart, `A = [(0,0,0),(1,0,0)]` and `B = [(0,1,0),(1,1,0)]`, give `2`.

The reference cell includes a nearest-neighbor helper so it runs alone. Your attempt can define its own helper in the same cell.


The PyTorch version has to stay differentiable. The check calls `.backward()` on the scalar. `torch.cdist` is fine. An in-place update of a tensor that requires grad is not.


In [ ]:
import torch

def chamfer_distance(A, B):
    """
    Sum of the two directed mean nearest-neighbor distances.

    mean_a min_b ||a-b|| + mean_b min_a ||b-a||.
    Some papers divide by 2. Say which definition you are using.
    """
    raise NotImplementedError


### Solution

Reference answer and checks. It replaces whatever you defined above.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_default_dtype(torch.float64)
DTYPE = torch.float64

def _t(x, dtype=DTYPE):
    if isinstance(x, torch.Tensor):
        return x.to(dtype=dtype)
    return torch.as_tensor(x, dtype=dtype)

def nearest_neighbors(A, B):
    """Nearest neighbor in B for every point of A.

    A: (N, D), B: (M, D)
    returns idx (N,) into B, and Euclidean distances (N,).
    torch.cdist is the usual PyTorch spelling of the broadcast distance.
    Memory is still O(N*M).
    """
    A = _t(A)
    B = _t(B)
    if B.shape[0] == 0:
        raise ValueError("B is empty")
    dist = torch.cdist(A, B)
    idx = dist.argmin(dim=1)
    return idx, dist[torch.arange(A.shape[0]), idx]

def chamfer_distance(A, B):
    """mean_a min_b ||a-b|| + mean_b min_a ||b-a||.

    Supports autograd. A and B are (N, 3) and (M, 3).
    """
    _, d_ab = nearest_neighbors(A, B)
    _, d_ba = nearest_neighbors(B, A)
    return d_ab.mean() + d_ba.mean()

def check_chamfer_distance():
    A = torch.tensor([[0.0, 0.0, 0.0], [1.0, 0.0, 0.0]])
    B = torch.tensor([[0.0, 1.0, 0.0], [1.0, 1.0, 0.0]])
    assert torch.allclose(chamfer_distance(A, B), torch.tensor(2.0))
    A = torch.tensor([[0.0, 0.0, 0.0], [10.0, 0.0, 0.0]])
    B = torch.tensor([[0.0, 0.0, 0.0]])
    assert torch.allclose(chamfer_distance(A, B), torch.tensor(5.0))
    assert torch.allclose(
        chamfer_distance(torch.tensor([[0.0, 0.0, 0.0]]), torch.tensor([[3.0, 4.0, 0.0]])),
        torch.tensor(10.0),
    )
    leaf = torch.tensor([[0.0, 0.0, 0.0], [1.0, 0.2, 0.0]], requires_grad=True)
    chamfer_distance(leaf, torch.tensor([[0.0, 1.0, 0.0], [2.0, 0.0, 0.0]])).backward()
    assert leaf.grad is not None and torch.isfinite(leaf.grad).all()
    print("passed")

check_chamfer_distance()


## 4. Voxel downsample

Replace each occupied voxel by the centroid of the points inside it. `voxel` is the cubic edge length. Return shape `(V, 3)`. Order does not matter.

This has to be correct for negative coordinates. A point at `(-0.1, 0, 0)` with `voxel = 1` is not in the same cell as `(0.1, 0, 0)`.

Hand check those two points, plus a second point in the positive cell, plus `(1.2, 0, 0)`. You should get three centroids, and the positive cell near the origin should be the **mean** of its points, not the last point written.

<details><summary>Hint</summary>

`torch.floor(points / voxel).to(torch.int64)` bins negative numbers toward -∞. Casting with `.to(torch.int64)` on a float truncates toward 0 and merges `-0.1` with `+0.1`. Keys come from `torch.unique(..., dim=0, return_inverse=True)`. Accumulate with `scatter_add_`. The statement `sums[inverse] += points` silently drops duplicate indices.

</details>


In [ ]:
import torch

def voxel_downsample(points, voxel):
    """
    One centroid per occupied voxel of cubic size `voxel`.

    points: (N, 3). Returns (V, 3). Empty input returns shape (0, 3).
    Row order does not matter. The result must be correct for negative coordinates.
    """
    raise NotImplementedError


### Solution

Reference answer and checks. It replaces whatever you defined above.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_default_dtype(torch.float64)
DTYPE = torch.float64

def _t(x, dtype=DTYPE):
    if isinstance(x, torch.Tensor):
        return x.to(dtype=dtype)
    return torch.as_tensor(x, dtype=dtype)

def voxel_downsample(points, voxel):
    """One centroid per occupied voxel. Correct for negative coordinates.

    torch.unique(..., dim=0, return_inverse=True) groups the keys.
    scatter_add_ accumulates points. sums[inverse] += points drops repeats.
    """
    points = _t(points)
    if points.shape[0] == 0:
        return torch.zeros((0, points.shape[1]), dtype=points.dtype)
    keys = torch.floor(points / voxel).to(torch.int64)
    _, inverse = torch.unique(keys, dim=0, return_inverse=True)
    counts = torch.bincount(inverse)
    sums = torch.zeros((counts.shape[0], points.shape[1]), dtype=points.dtype)
    sums.scatter_add_(0, inverse[:, None].expand_as(points), points)
    return sums / counts[:, None]

def _same_rows(got, expected, atol=1e-8):
    got = _t(got)
    expected = _t(expected)
    assert got.shape == expected.shape, (got.shape, expected.shape)
    order = torch.argsort(got[:, 0] * 1e6 + got[:, 1] * 1e3 + got[:, 2])
    order_e = torch.argsort(expected[:, 0] * 1e6 + expected[:, 1] * 1e3 + expected[:, 2])
    assert torch.allclose(got[order], expected[order_e], atol=atol)

def check_voxel_downsample():
    pts = torch.tensor(
        [
            [0.1, 0.1, 0.1],
            [0.2, 0.2, 0.4],
            [-0.1, 0.0, 0.0],
            [1.2, 0.0, 0.0],
        ]
    )
    got = voxel_downsample(pts, 1.0)
    expected = torch.tensor(
        [
            [-0.1, 0.0, 0.0],
            [0.15, 0.15, 0.25],
            [1.2, 0.0, 0.0],
        ]
    )
    _same_rows(got, expected)
    assert got.shape[0] == 3
    print("passed")

check_voxel_downsample()
